In [ ]:
import os, subprocess
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_req = f"/Workspace{os.path.dirname(os.path.dirname(_nb))}/requirements-ingestion.txt"
subprocess.check_call(["pip", "install", "-q", "-r", _req])
dbutils.library.restartPython()

# Ask Jorge — Document Ingestion

Extracts text from PDF files, cleans it, chunks it by token count, and writes
chunks to `jorge.cv_rag.jorge_cv_chunks` (Delta + CDF enabled).

**Parameters:**
- `file_path` — full Volume path to a single PDF; leave empty to process all PDFs in `volume_path`
- `volume_path` — Volume directory scanned when `file_path` is empty (default: `/Volumes/jorge/cv_rag/jorge_cv_docs`)
- `full_reindex` — `'true'` truncates the whole table before writing; `'false'` replaces only chunks for this file

In [ ]:
import os
import re
import unicodedata
import time
import random
import glob
from datetime import datetime, timezone
from typing import List, Tuple

import pypdf
from transformers import AutoTokenizer

TABLE_NAME      = "jorge.cv_rag.jorge_cv_chunks"
# Volume-based cache avoids re-downloading on every run and survives cluster restarts
TOKENIZER_CACHE = "/Volumes/jorge/cv_rag/hf_cache"
MAX_TOKENS      = 500
OVERLAP_TOKENS  = 50   # tokens shared between consecutive chunks for context continuity
MIN_CHUNK_TOKENS = 30  # discard fragments shorter than this
SEPARATORS      = ["\n\n", "\n", ". ", " ", ""]
LLM_TRANSLATION_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"


In [ ]:
dbutils.widgets.text("file_path",   "")
dbutils.widgets.text("volume_path",  "/Volumes/jorge/cv_rag/jorge_cv_docs")
dbutils.widgets.text("full_reindex", "true")

file_path    = dbutils.widgets.get("file_path").strip()
volume_path  = dbutils.widgets.get("volume_path").strip()
full_reindex = dbutils.widgets.get("full_reindex").lower() == "true"

# Resolve files to process
if file_path:
    files_to_process = [file_path]
elif full_reindex:
    files_to_process = sorted(glob.glob(f"{volume_path}/*.pdf"))
    if not files_to_process:
        raise ValueError(f"No PDF files found in volume_path={volume_path!r}")
    print(f"full_reindex mode: {len(files_to_process)} files found in {volume_path}")
else:
    raise ValueError("file_path is required when full_reindex=False")

print(f"file_path:    {file_path or '(all)'}")
print(f"volume_path:  {volume_path}")
print(f"full_reindex: {full_reindex}")
print(f"files:        {files_to_process}")

In [ ]:
def clean_text(text: str) -> str:
    """Normalise and clean raw PDF text."""
    # Unicode normalisation (e.g. ligatures: ﬁ → fi)
    text = unicodedata.normalize("NFKC", text)
    # Remove BPE special tokens that leak into some PDFs
    text = re.sub(r"</?s>", "", text)
    # Non-breaking spaces → regular space
    text = text.replace("\xa0", " ")
    # Rejoin words split by hyphen at end-of-line
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    # Collapse runs of spaces/tabs (keep newlines for separator logic)
    text = re.sub(r"[ \t]+", " ", text)
    # Collapse 3+ blank lines into 2
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Remove lone page numbers (lines with only digits)
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)
    # Strip leading/trailing whitespace per line
    text = "\n".join(line.strip() for line in text.splitlines())
    return text.strip()


def extract_pages(pdf_path: str) -> List[Tuple[int, str]]:
    """Return list of (page_number, cleaned_text) tuples."""
    pages = []
    try:
        with open(pdf_path, "rb") as f:
            reader = pypdf.PdfReader(f)
            for i, page in enumerate(reader.pages, start=1):
                raw = page.extract_text() or ""
                cleaned = clean_text(raw)
                if cleaned:
                    pages.append((i, cleaned))
    except Exception as e:
        raise RuntimeError(f"Failed to read {pdf_path}: {e}") from e
    return pages


def chunk_page(text: str, tokenizer, page_num: int) -> List[dict]:
    """
    Split a page's text into overlapping chunks under MAX_TOKENS.
    Returns list of dicts with chunk_text, page_number, chunk_index, chunk_tokens.
    """
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    idx = 0
    while start < len(tokens):
        end = min(start + MAX_TOKENS, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True).strip()
        if len(chunk_tokens) >= MIN_CHUNK_TOKENS and chunk_text:
            chunks.append({
                "chunk_text":   chunk_text,
                "page_number":  page_num,
                "chunk_index":  idx,
                "chunk_tokens": len(chunk_tokens),
            })
            idx += 1
        if end == len(tokens):
            break
        # Advance by (MAX_TOKENS - OVERLAP_TOKENS) to create overlap
        start += MAX_TOKENS - OVERLAP_TOKENS
    return chunks


import mlflow.deployments as _deploy

def translate_to_english(text: str) -> str:
    """Translate a chunk to English using the Databricks LLM endpoint."""
    client = _deploy.get_deploy_client("databricks")
    response = client.predict(
        endpoint=LLM_TRANSLATION_ENDPOINT,
        inputs={
            "messages": [{"role": "user", "content": f"Translate the following text to English. Reply ONLY with the translation, preserving the structure:\n\n{text}"}],
            "temperature": 0,
            "max_tokens": 600,
        },
    )
    return response["choices"][0]["message"]["content"].strip()


def ensure_table():
    """Create or migrate the chunks table."""
    table_exists = False
    try:
        spark.sql(f"DESCRIBE TABLE {TABLE_NAME}")
        table_exists = True
    except Exception:
        pass

    if table_exists and full_reindex:
        print(f"full_reindex=True: dropping and recreating {TABLE_NAME}")
        spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")
        table_exists = False
    elif table_exists:
        # Incremental: add chunk_text_es if missing (schema migration)
        try:
            spark.sql(f"ALTER TABLE {TABLE_NAME} ADD COLUMNS (chunk_text_es STRING)")
            print(f"Added chunk_text_es column to {TABLE_NAME}.")
        except Exception:
            pass  # Column already exists
        spark.sql(
            f"ALTER TABLE {TABLE_NAME} "
            "SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )

    if not table_exists:
        print(f"Creating {TABLE_NAME}...")
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
                id             BIGINT GENERATED BY DEFAULT AS IDENTITY,
                chunk_text     STRING  NOT NULL,
                chunk_text_es  STRING,
                source_file    STRING  NOT NULL,
                page_number    INT,
                chunk_index    INT,
                chunk_tokens   INT,
                ingested_at    TIMESTAMP
            )
            TBLPROPERTIES (delta.enableChangeDataFeed = true)
        """)
        print(f"Table {TABLE_NAME} created.")
)

In [ ]:
def load_tokenizer(cache_dir: str, max_retries: int = 3) -> AutoTokenizer:
    """Load tokenizer with retries to handle concurrent cache-lock scenarios."""
    #os.makedirs(cache_dir, exist_ok=True)
    for attempt in range(max_retries):
        try:
            return AutoTokenizer.from_pretrained(
                "hf-internal-testing/llama-tokenizer",
                cache_dir=cache_dir,
            )
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 15 + random.randint(0, 15)
                print(f"Tokenizer load failed (attempt {attempt + 1}/{max_retries}): {e}")
                print(f"Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


print("Loading tokenizer...")
tokenizer = load_tokenizer(TOKENIZER_CACHE)

ensure_table()

grand_total_chunks = 0
for fp in files_to_process:
    filename = os.path.basename(fp)
    print(f"\nExtracting pages from {filename}...")
    pages = extract_pages(fp)
    if not pages:
        print(f"  WARNING: no text extracted from {fp} — skipping.")
        continue
    print(f"  {len(pages)} pages extracted.")

    # For incremental mode, replace only this file's chunks
    if not full_reindex:
        print(f"  Deleting existing chunks for {filename}...")
        spark.sql(f"DELETE FROM {TABLE_NAME} WHERE source_file = '{filename}'")

    raw_chunks = []
    now = datetime.now(timezone.utc)
    for page_num, page_text in pages:
        page_chunks = chunk_page(page_text, tokenizer, page_num)
        for c in page_chunks:
            raw_chunks.append((
                c["chunk_text"],
                filename,
                c["page_number"],
                c["chunk_index"],
                c["chunk_tokens"],
                now,
            ))

    # Translate each chunk to English for the English-only embedding model
    print(f"  Translating {len(raw_chunks)} chunks to English...")
    all_chunks = []
    for i, c in enumerate(raw_chunks):
        chunk_text_es = c[0]
        chunk_text_en = translate_to_english(chunk_text_es)
        all_chunks.append((
            chunk_text_en,  # chunk_text (English — used for embedding)
            chunk_text_es,  # chunk_text_es (original Spanish — kept for reference)
            c[1], c[2], c[3], c[4], c[5],
        ))
        if (i + 1) % 5 == 0 or (i + 1) == len(raw_chunks):
            print(f"    {i + 1}/{len(raw_chunks)} chunks translated.")

    schema = "chunk_text STRING, chunk_text_es STRING, source_file STRING, page_number INT, chunk_index INT, chunk_tokens INT, ingested_at TIMESTAMP"
    (
        spark.createDataFrame(all_chunks, schema=schema)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLE_NAME)
    )
    print(f"  {len(all_chunks)} chunks written for {filename}.")
    grand_total_chunks += len(all_chunks)

total = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_NAME}").collect()[0]["n"]
print(f"\nDone. {grand_total_chunks} chunks written across {len(files_to_process)} file(s). Table total: {total} chunks.")